Train classifier of primary endpoint type in EUCT-NS dataset using CNB with TF-IDF Features

In [ ]:
import pandas as pd

import numpy as np 
from numpy import mean, std

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import ComplementNB 
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, classification_report, balanced_accuracy_score
import joblib

import nltk

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import tqdm as notebook_tqdm

import joblib

Train the classifier

In [ ]:
second_ds = pd.read_csv(second_ds)
second_ds['concat_corpus'] = second_ds['Title']+ " " + second_ds['Objective'] + " " + second_ds['1ry_endpoint'] 
second_ds['concat_corpus'] = second_ds['concat_corpus'].fillna('')

Load saved model and embeddings from training the original classifier

In [ ]:
embedded_transformed_features = joblib.load("feature_embeddings.pkl")
X2 = embedded_transformed_features.transform(second_ds['concat_corpus'])

mdl = joblib.load('model.pkl')
# Load the trained model
model.predict(X2)
y_pred = model.predict(X2)


In [ ]:
feature_names = vectorizer.get_feature_names_out()
X2_df = pd.DataFrame(X2.toarray(), columns=feature_names)

In [ ]:
y_pred = model.predict(X2_df)
# To fix the warning about DataFrame input

In [ ]:
unique, counts = np.unique(y_pred, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
confidences = model.predict_proba(X2_df).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
surrogate_confidence_class_2 = model.predict_proba(X2_df.iloc[[2]])[0][2]
print(f"Confidence for predicting class 2: {surrogate_confidence_class_2:.2f}")

In [ ]:
low_confidence_indices = np.where(confidences < 0.6)[0]
print(f"Low confidence predictions: {len(low_confidence_indices)}")

Active learning: Warm Start

1. Least confidence sampling

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_df.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y_initial = manually_label_samples(X_initial_raw)
X_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
# Ensure X_train is always a list before extending
if 'X_train' not in locals() or not isinstance(X_train, list):
    X_train = list(X_initial)  # Convert to list explicitly
    y_train = np.array(y_initial)
else:
    X_train.extend(X_initial)
    y_train = np.hstack([y_train, y_initial])

# Remove labeled samples from unlabeled pool
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
print(len(X_train), len(y_train), X2_unlabeled_reset.shape)

Training

In [ ]:
pipeline = make_pipeline(
    TfidfVectorizer(analyzer='word', min_df=3, ngram_range=(1, 3)),
    ComplementNB(alpha=0.1, fit_prior=True)
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X2_unlabeled_reset)

In [ ]:
from sklearn.model_selection import cross_val_predict

y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=3)

In [ ]:
iter1 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter1)

Iteration 2

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1) # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y2_initial = manually_label_samples(X_initial_raw)
X2_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X2_initial = list(X2_initial)
y_train = np.array(y2_initial)

In [ ]:
print(len(X2_initial))

In [ ]:
print(len(X_train))

In [ ]:
X_train = X_train + X2_initial
y_train = np.hstack([y_initial, y2_initial])
len(X_train), len(y_train)

In [ ]:
# Remove labelled samples from unlabelled pool
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X2_unlabeled_reset)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=3)

In [ ]:
iter2 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter2)

Iteration 3

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y3_initial = manually_label_samples(X_initial_raw)
X3_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X3_initial = list(X3_initial)
y_train = np.array(y3_initial)

In [ ]:
X_train = X_train + X3_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=3)

In [ ]:
iter3 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter3)

Iteration 4

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y4_initial = manually_label_samples(X_initial_raw)
X4_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X4_initial = list(X4_initial)
y_train = np.array(y4_initial)

In [ ]:
X_train = X_train + X4_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter4 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter4)

Iteration 5

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y5_initial = manually_label_samples(X_initial_raw)
X5_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X5_initial = list(X5_initial)
y_train = np.array(y5_initial)

In [ ]:
X_train = X_train + X5_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter5 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter5)

Iteration 6

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y6_initial = manually_label_samples(X_initial_raw)
X6_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X6_initial = list(X6_initial)
y_train = np.array(y6_initial)

In [ ]:
X_train = X_train + X6_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter6 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter6)

Iteration 7

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y7_initial = manually_label_samples(X_initial_raw)
X7_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X7_initial = list(X7_initial)
y_train = np.array(y7_initial)

In [ ]:
X_train = X_train + X7_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter7 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter7)

Iteration 8

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y8_initial = manually_label_samples(X_initial_raw)
X8_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X8_initial = list(X8_initial)
y_train = np.array(y8_initial)

In [ ]:
X_train = X_train + X8_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter8 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter8)

Iteration 9

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y9_initial = manually_label_samples(X_initial_raw)
X9_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X9_initial = list(X9_initial)
y_train = np.array(y9_initial)

In [ ]:
X_train = X_train + X9_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter9 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter9)

Iteration 10

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y10_initial = manually_label_samples(X_initial_raw)
X10_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X10_initial = list(X10_initial)
y_train = np.array(y10_initial)

In [ ]:
X_train = X_train + X10_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter10 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter10)

Iteration 11

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y11_initial = manually_label_samples(X_initial_raw)
X11_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X11_initial = list(X11_initial)
y_train = np.array(y11_initial)

In [ ]:
X_train = X_train + X11_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=5)

iter11 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter11)

Iteration 12

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y12_initial = manually_label_samples(X_initial_raw)
X12_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X12_initial = list(X12_initial)
y_train = np.array(y12_initial)

In [ ]:
X_train = X_train + X12_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter12 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter12)

Iteration 13

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y13_initial = manually_label_samples(X_initial_raw)
X13_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X13_initial = list(X13_initial)
y_train = np.array(y13_initial)

In [ ]:
X_train = X_train + X13_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter13 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter13)

Iteration 14

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y14_initial = manually_label_samples(X_initial_raw)
X14_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X14_initial = list(X14_initial)
y_train = np.array(y14_initial)

In [ ]:
X_train = X_train + X14_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial, y14_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter14 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter14)

Iteration 15

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y15_initial = manually_label_samples(X_initial_raw)
X15_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X15_initial = list(X15_initial)
y_train = np.array(y15_initial)

In [ ]:
X_train = X_train + X15_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial, y14_initial, y15_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter15 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter15)

Iteration 16

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y16_initial = manually_label_samples(X_initial_raw)
X16_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X16_initial = list(X16_initial)
y_train = np.array(y16_initial)

In [ ]:
X_train = X_train + X16_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial, y14_initial, y15_initial, y16_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter16 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter16)

Iteration 17

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y17_initial = manually_label_samples(X_initial_raw)
X17_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X17_initial = list(X17_initial)
y_train = np.array(y17_initial)

In [ ]:
X_train = X_train + X17_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial, y14_initial, y15_initial, y16_initial, y17_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter17 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter17)

Iteration 18

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y18_initial = manually_label_samples(X_initial_raw)
X18_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X18_initial = list(X18_initial)
y_train = np.array(y18_initial)

In [ ]:
X_train = X_train + X18_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial, y14_initial, y15_initial, y16_initial, y17_initial, y18_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter18 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter18)

Iteration 19

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y19_initial = manually_label_samples(X_initial_raw)
X19_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X19_initial = list(X19_initial)
y_train = np.array(y19_initial)

In [ ]:
X_train = X_train + X19_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial, y14_initial, y15_initial, y16_initial, y17_initial, y18_initial, y19_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter19 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter19)

Iteration 20

In [ ]:
# Define the unlabeled dataset
X2_unlabeled = X2_unlabeled_reset.copy()

# Obtain probabilities and calculate uncertainty scores
probs = model.predict_proba(X2_unlabeled)
uncertainty_scores = np.abs(probs - 0.5).min(axis=1)  # Least confidence sampling

# Select the top N most uncertain samples
n_samples = 7
most_uncertain_indices = uncertainty_scores.argsort()[-n_samples:]

# Extract the corresponding Unique_IDs and preprocessed concatenated text
X_initial_raw = ns_hra.iloc[most_uncertain_indices]

def manually_label_samples(selected_rows):
    labels = []
    for idx, row in selected_rows.iterrows():
        print(f"Unique_ID: {row['Unique_ID']}")
        print(f"Preprocessed Text: {row['concat_corpus']}\n")
        label = input("Enter label for this sample (e.g., 0, 1, 2): ")
        labels.append(int(label))
    return labels

# Manually label the selected samples
y20_initial = manually_label_samples(X_initial_raw)
X20_initial = X_initial_raw['concat_corpus'].tolist()

In [ ]:
X20_initial = list(X20_initial)
y_train = np.array(y20_initial)

In [ ]:
X_train = X_train + X20_initial
y_train = np.hstack([y_initial, y2_initial, y3_initial, y4_initial, y5_initial, y6_initial, y7_initial, y8_initial, y9_initial, y10_initial, y11_initial, y12_initial, y13_initial, y14_initial, y15_initial, y16_initial, y17_initial, y18_initial, y19_initial, y20_initial])
len(X_train), len(y_train)

In [ ]:
X2_unlabeled_reset = X2_unlabeled.drop(index=most_uncertain_indices).reset_index(drop=True)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
confidences = pipeline.predict_proba(X2_unlabeled_reset).max(axis=1)
print(f"Average confidence: {np.mean(confidences):.2f}")

In [ ]:
y_pred_cv = cross_val_predict(pipeline, X_train, y_train, cv=10)

iter20 = classification_report(y_train, y_pred_cv, output_dict=True)
print(iter20)

Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Step 1: Gather all classification reports
iterations = 20
classification_reports = [globals()[f'iter{i}'] for i in range(1, iterations+1)]

# Step 2: Extract metrics
accuracy_list = []
f1_score_0 = []
f1_score_1 = []
f1_score_2 = []

for report in classification_reports:
    accuracy_list.append(report['accuracy'])
    f1_score_0.append(report['0']['f1-score'])
    f1_score_1.append(report['1']['f1-score'])
    f1_score_2.append(report['2']['f1-score'])

iterations = np.arange(1, 21)

# Step 3: Plot accuracy
plt.figure(figsize=(10, 5))
plt.plot(iterations, accuracy_list, marker='o', linestyle='-', color='b', label='Accuracy')
plt.axhline(y=0.3632, color='r', linestyle='--', label='Baseline Accuracy')
plt.xlabel('Iteration')
plt.ylabel('Accuracy')
plt.title('Accuracy vs. Iteration')
plt.ylim(0, 1)
plt.legend()
plt.grid()
plt.show()

# Step 4: Plot F1 scores per class
plt.figure(figsize=(10, 5))
plt.plot(iterations, f1_score_0, marker='o', linestyle='-', color='r', label='F1-score Class 0 (PFO)')
plt.plot(iterations, f1_score_1, marker='s', linestyle='--', color='g', label='F1-score Class 1 (IO)')
plt.plot(iterations, f1_score_2, marker='^', linestyle='-.', color='m', label='F1-score Class 2 (SO)')
plt.axhline(y=0.3830, color='r', linestyle='--', label='Baseline Weighted F1-score')
plt.xlabel('Iteration')
plt.ylabel('F1-Score')
plt.title('F1-Score per Class vs. Iteration')
plt.ylim(0, 1)
plt.legend()
plt.grid()
plt.show()


Precision recall curve and threshold selection

In [ ]:
#Troubleshooting
print(f"y_true_bin shape: {y_true_bin.shape}")
print(f"y_score shape: {y_score.shape}")

In [ ]:
y_pred_proba = pipeline.predict_proba(X_train)

In [ ]:
print(f"y_pred_proba shape: {y_pred_proba.shape}")
print(f"y_train shape: {y_train.shape}")

In [ ]:
from sklearn.metrics import precision_recall_curve

# Target recall threshold
desired_recall = 0.70
selected_thresholds = {}

# For each class (assuming classes 0, 1, 2)
classes = [0, 1, 2]

for cls in classes:
    print(f"\n--- Class {cls} ---")
    
    # Binary ground truth: 1 if current class, 0 otherwise
    y_true_bin = (y_train == cls).astype(int)
    
    # Predicted probability scores for the current class
    y_score = y_pred_proba[:, cls]
    
    # Compute precision-recall pairs
    precision, recall, thresholds = precision_recall_curve(y_true_bin, y_score)
    
    # precision_recall_curve gives (n_thresholds + 1) recall/precision points
    # thresholds has shape (n_thresholds,)
    
    # Find index where recall crosses desired_recall
    try:
        index = np.where(recall >= desired_recall)[0][0]
        
        if index >= len(thresholds):
            # Edge case: the last recall value doesn't have a corresponding threshold
            selected_threshold = thresholds[-1]
        else:
            selected_threshold = thresholds[index]
        
        selected_thresholds[cls] = selected_threshold
        print(f"Threshold @ Recall ≥ {desired_recall:.2f}: {selected_threshold:.4f}")
        print(f"  Recall: {recall[index]:.2f}, Precision: {precision[index]:.2f}")
        
    except IndexError:
        print(f"No threshold found for class {cls} reaching recall ≥ {desired_recall}")
        selected_thresholds[cls] = None
    
    # Plot the Precision-Recall curve
    plt.figure()
    plt.plot(recall, precision, marker='.', label=f'Class {cls}')
    plt.axvline(x=desired_recall, color='red', linestyle='--', label=f'Target Recall = {desired_recall:.2f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curve for Class {cls}')
    plt.legend()
    plt.grid(True)
    plt.show()

print("\nSelected thresholds per class:")
for cls, thresh in selected_thresholds.items():
    print(f"Class {cls}: {thresh}")


Biasing CNB for class 2

In [ ]:
threshold_class_2 = selected_thresholds[2]

# New adjusted predictions
adjusted_preds = []

for prob in y_pred_proba:
    if prob[2] >= threshold_class_2:
        adjusted_preds.append(2)  # Force predict class 2
    else:
        adjusted_preds.append(np.argmax(prob))  # Otherwise, normal prediction

adjusted_preds = np.array(adjusted_preds)


In [ ]:
print(classification_report(y_train, adjusted_preds))

In [ ]:
# Plotting Precision and Recall vs Threshold for Class 2
thresholds = np.linspace(0, 1, 100)

# Store precision and recall at each threshold
precisions = []
recalls = []

for thresh in thresholds:
    adjusted_preds = []
    
    for prob in y_pred_proba:
        if prob[2] >= thresh:
            adjusted_preds.append(2)
        else:
            adjusted_preds.append(np.argmax(prob))
    
    adjusted_preds = np.array(adjusted_preds)
    
    # Precision and recall class 2
    precision = precision_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    recall = recall_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    
    precisions.append(precision)
    recalls.append(recall)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions, label='Precision', color='blue')
plt.plot(thresholds, recalls, label='Recall', color='green')
plt.axhline(0.7, color='red', linestyle='--', label='Target Recall = 0.70')
plt.axvline(threshold_class_2, color='purple', linestyle='--', label=f'Selected Threshold = {threshold_class_2:.2f}')
plt.xlabel('Threshold for Class 2')
plt.ylabel('Score')
plt.title('Precision and Recall vs Threshold (Class 2)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
thresholds = np.linspace(0, 1, 500)

best_f1 = 0
best_threshold = 0
best_precision = 0
best_recall = 0

for thresh in thresholds:
    adjusted_preds = []
    
    for prob in y_pred_proba:
        if prob[2] >= thresh:
            adjusted_preds.append(2)
        else:
            adjusted_preds.append(np.argmax(prob))
    
    adjusted_preds = np.array(adjusted_preds)
    
    # Calculate precision, recall, f1 for class 2 only
    precision = precision_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    recall = recall_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    f1 = f1_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh
        best_precision = precision
        best_recall = recall

print(f"Best threshold for Class 2 = {best_threshold:.4f}")
print(f"  F1-score = {best_f1:.4f}")
print(f"  Precision = {best_precision:.4f}")
print(f"  Recall = {best_recall:.4f}")

In [ ]:
f1_scores = []

for thresh in thresholds:
    adjusted_preds = []
    
    for prob in y_pred_proba:
        if prob[2] >= thresh:
            adjusted_preds.append(2)
        else:
            adjusted_preds.append(np.argmax(prob))
    
    adjusted_preds = np.array(adjusted_preds)
    
    f1 = f1_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    f1_scores.append(f1)

# Plot F1 vs Threshold
plt.figure(figsize=(10,6))
plt.plot(thresholds, f1_scores, color='blue')
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best Threshold = {best_threshold:.2f}')
plt.xlabel('Threshold for Class 2')
plt.ylabel('F1-score (Class 2)')
plt.title('F1-Score vs Threshold (Class 2)')
plt.grid(True)
plt.legend()
plt.show()

Post-Hoc threshold analysis

In [ ]:
desired_recall = 0.7  
thresholds = np.linspace(0, 1, 500)

# To store candidates
valid_thresholds = []
valid_precisions = []
valid_recalls = []
valid_f1s = []

for thresh in thresholds:
    adjusted_preds = []
    
    for prob in y_pred_proba:
        if prob[2] >= thresh:
            adjusted_preds.append(2)
        else:
            adjusted_preds.append(np.argmax(prob))
    
    adjusted_preds = np.array(adjusted_preds)
    
    # Evaluate class 2 only
    precision = precision_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    recall = recall_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    f1 = f1_score(y_train, adjusted_preds, labels=[2], average='micro', zero_division=0)
    
    # Only accept thresholds meeting the recall constraint
    if recall >= desired_recall:
        valid_thresholds.append(thresh)
        valid_precisions.append(precision)
        valid_recalls.append(recall)
        valid_f1s.append(f1)

# Select best precision among valid thresholds
if valid_thresholds:
    best_idx = np.argmax(valid_precisions)
    best_threshold = valid_thresholds[best_idx]
    best_precision = valid_precisions[best_idx]
    best_recall = valid_recalls[best_idx]
    best_f1 = valid_f1s[best_idx]
    
    print(f"Best threshold (recall ≥ {desired_recall}): {best_threshold:.4f}")
    print(f"  Precision: {best_precision:.4f}")
    print(f"  Recall: {best_recall:.4f}")
    print(f"  F1-score: {best_f1:.4f}")
else:
    print(f"No threshold found achieving recall ≥ {desired_recall}.")


In [ ]:
if valid_thresholds:
    plt.figure(figsize=(10,6))
    plt.plot(valid_thresholds, valid_precisions, marker='o', label='Precision')
    plt.plot(valid_thresholds, valid_recalls, marker='x', label='Recall')
    plt.plot(valid_thresholds, valid_f1s, marker='^', label='F1-score')
    plt.axhline(desired_recall, color='red', linestyle='--', label=f'Target Recall = {desired_recall}')
    plt.xlabel('Threshold for Class 2')
    plt.ylabel('Score')
    plt.title('Valid Thresholds (Recall ≥ Target)')
    plt.legend()
    plt.grid(True)
    plt.show()
